# 固定09:29条件NYSE四日日历代理

阶段1为source-feature与train-only无拟合预检，阶段2为全部6模型先固定后评价。当前仅实现/synthetic已完成，实际预检与拟合/账户均未获GO。

In [ ]:
import os, sys, json
from pathlib import Path
root = Path.cwd()
while not (root / "AGENTS.md").exists():
    if root.parent == root: raise RuntimeError("workspace not found")
    root = root.parent
sys.path.insert(0,str(root))
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
import qlib
from qlib.workflow import R
from qlib.data.dataset import DatasetH
card = root / "research/stage1_preopen_tom_20261018"
plan = json.loads((card / "plan.json").read_text())
print({"plan_status":plan["status"],"no_data_loaded_in_setup":True})

## 独立授权的执行阶段

prepare只读取已批准calendar primitives和完整train-only对象，不读取evaluation labels/returns。所有6finalmodels固定后run才解锁评价Dataset与原账户输入；两个环境开关互斥，均不能替代对应独评及owner GO。Dataset保存前dump_all recursive，最终直接prepare读回。

In [ ]:
from IPython.display import Markdown, display
from src.workflow.preopen_tom_allocation import prepare, run
preflight_mode = os.environ.get("PREOPEN_TOM_PREFLIGHT") == "1"
execution_mode = os.environ.get("PREOPEN_TOM_EXECUTE") == "1"
if preflight_mode and execution_mode:
    raise RuntimeError("Stage switches are mutually exclusive")
if preflight_mode:
    result = prepare(root=root,card=card)
    display(Markdown((card / "conclusion.md").read_text()))
elif execution_mode:
    result = run(root=root,card=card)
    display(Markdown((card / "conclusion.md").read_text()))
else:
    print("Implementation only: no actual source-feature, training-data, fit or account execution.")

# 09:29条件NYSE日历代理：实现阶段

已实现固定四日条件代理、B/C四槽模型适配和阶段隔离工作流。三个集中synthetic合同均通过：feature71154、model3495、训练驻留范围33079，各1 passed；没有真实source/训练对象读取或proxy生成，0实际no-fit预检、fit、账户和新来源HTTP。

训练白名单只用已封存PG训练专用Dataset3014×85及20741×20，保留全部原列并只追加4列。未来预检将核全部驻留handler/loader帧、原39/16及11tensor/scaler、共同cohort和三mask组内proxy0/1变化。评价标签/收益对象仅在6个最终模型SHA固定后才加载；不能先加载全期对象再crop称未读。

原allocation收益为H1次开盘到下次开盘，tail/alpha/cash为H5；共同H5成熟期筛选保持3014条，不因H1较早成熟扩样本。此为语义澄清，无标签或数学变更。

未来上限为6fits、6个等权ensemble费用原生账户、15旧参考，共21books及15CI，当前均未执行。没有新的收益/窗口结果，不晋升。来源仍conditional/PITfalse，matched B的三mask本身含部分日历信息；数值增量是否可辨识须先过冻结预检门。

下一步：独立整体实现review与owner source-feature+train-only no-fit GO；实际预检通过后，才由合并最终design/preflight-result评审和ownerfit/account GO批准正式实验。

入口：[plan.json](plan.json)、[source_whitelist.json](source_whitelist.json)、[implementation_identity.json](implementation_identity.json)、[label_horizon_clarification.json](label_horizon_clarification.json)。
